In [1]:
import re
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook
from datetime import datetime

def find_all_sheet(workbook):
    """Finds the 'All' tab regardless of case variations (ALL, All, all)."""
    for sheet_name in workbook.sheetnames:
        if sheet_name.lower().strip() == "all":
            return workbook[sheet_name]
    return None

In [2]:
def extract_cell_value_and_color(cell):
    """
    Extracts text value and checks if senior staff highlighted the cell.
    Converts visual shapes or specific markers to standard text strings.
    """
    val = cell.value
    if val is None:
        return None, False
        
    # Standardize string formatting
    val_str = str(val).strip().lower()
    
    # Map symbols to clear database strings (Modify these characters if needed)
    if val_str in ["○", "circle", "o"]:
        availability = "online & in person"
    elif "online" in val_str:
        availability = "online"
    elif "in person" in val_str or "in-person" in val_str:
        availability = "in person"
    else:
        availability = str(val).strip() # Fallback for 'X' or other text markers

    # Check for senior staff background color fills
    is_approved = False
    if cell.fill and cell.fill.start_color:
        color_hex = cell.fill.start_color.rgb
        # Filter out default uncolored grids (openpyxl often returns '00000000' or 0)
        if color_hex and color_hex != "00000000" and color_hex != 0:
            is_approved = True
            
    return availability, is_approved

In [3]:
def parse_time_block(time_str):
    """Splits a string range like '10:00-10:30' into separate clean time parts."""
    if not time_str or "-" not in str(time_str):
        return None, None
    try:
        parts = str(time_str).split("-")
        start = parts[0].strip() + ":00"
        end = parts[1].strip() + ":00"
        return start, end
    except Exception:
        return None, None

In [4]:
def find_grid_boundaries(sheet):
    """
    Locates the time column using a regex pattern. Then, scans to the left 
    to find the true Date column, ignoring any weekday text columns in between.
    """
    time_pattern = re.compile(r"^\d{1,2}:\d{2}-\d{1,2}:\d{2}$")
    
    # 1. Find the time column first
    for row_idx in range(1, 20):  # Scans up to row 20
        for col_idx in range(1, 15):  # Scans up to column 15
            cell_val = str(sheet.cell(row=row_idx, column=col_idx).value or "").strip()
            
            if time_pattern.match(cell_val):
                time_col_idx = col_idx
                student_header_row = row_idx - 1
                
                # 2. Scan BACKWARDS from the time column to locate the Date column
                date_col_idx = None
                for look_back in range(time_col_idx - 1, 0, -1):
                    test_val = sheet.cell(row=row_idx, column=look_back).value
                    
                    # If the cell is empty due to a vertical merge, we check rows above it
                    # within reasonable bounds to see if it belongs to a date column block.
                    check_row = row_idx
                    while test_val is None and check_row > student_header_row:
                        check_row -= 1
                        test_val = sheet.cell(row=check_row, column=look_back).value
                    
                    if test_val is not None:
                        test_str = str(test_val).strip().lower()
                        # Skip common day names and abbreviations
                        if test_str in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
                                        'mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun']:
                            continue
                        
                        # If it's not a day string, this is our true Date column anchor!
                        date_col_idx = look_back
                        break
                
                # Fallback safeguard: if no date column found, assume it's the immediate left
                if date_col_idx is None:
                    date_col_idx = time_col_idx - 1
                    
                return student_header_row, date_col_idx, time_col_idx
                
    raise ValueError("Could not find a valid time block (e.g. 9:00-9:30) to anchor the data grid.")

In [23]:
def process_all_workbooks(folder_path):
    """Main pipeline loop over all 19 files."""
    # Convert input string to a Path object
    master_records = []
    root_dir = Path(folder_path)
    
    if not root_dir.exists():
        print(f"[ERROR] The directory path '{folder_path}' does not exist.")
        return []
        
    excel_extensions = {'.xlsx', '.xlsm'}
    discovered_paths = []
    
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] starting recursive scan in: {root_dir.resolve()}")
    
    # rglob("*") recursively yields all files and directories under the root path
    for path in root_dir.rglob("*"):
        # Ensure it's a file, matches our target extensions, and isn't an Excel temporary/lock file
        if path.is_file() and path.suffix.lower() in excel_extensions:
            if not path.name.startswith("~$"):
                # Append the absolute path as a string to match your existing functions
                discovered_paths.append(str(path.resolve()))
                
    print(f"Scan complete. Discovered {len(discovered_paths)} potential Excel files across all subfolders.\n")
    
    for file_path in discovered_paths:
        print(f"Processing: {file_path}...")
        
        # Load workbook with data_only=True to get raw evaluation values instead of raw formulas
        wb = load_workbook(file_path, data_only=True)
        sheet = find_all_sheet(wb)
        
        if sheet is None:
            print(f" -> Skipping: No 'All' sheet found in {file_path}")
            continue
            
        try:
            student_row, date_col, time_col = find_grid_boundaries(sheet)            
        except ValueError as e:
            print(f" -> Skipping {file_path}: {e}")
            continue

        # Extract all student names across the header row starting after the time column
        students = {}
        max_col = sheet.max_column
        for c in range(time_col + 1, max_col + 1):
            name = sheet.cell(row=student_row, column=c).value
            if name:
                students[c] = str(name).strip()

        # Step through rows down the schedule grid
        max_row = sheet.max_row
        current_date = None # Track merged date across iterations
        
        for r in range(student_row + 1, max_row + 1):
            # Resolve merged or standalone cell value for Date
            date_cell = sheet.cell(row=r, column=date_col)
            # Handle openpyxl merged cell tracking lookup automatically
            if date_cell.coordinate in sheet.merged_cells:
                # Find the top-left cell of the merge range to extract value safely
                for merge_range in sheet.merged_cells.ranges:
                    if date_cell.coordinate in merge_range:
                        date_cell = sheet.cell(row=merge_range.min_row, column=merge_range.min_col)
                        break
            
            if date_cell.value is not None:
                # Format to database friendly YYYY-MM-DD
                try:
                    current_date = pd.to_datetime(date_cell.value).strftime('%Y-%m-%d')
                except Exception:
                    current_date = str(date_cell.value).strip()

            time_cell_val = str(sheet.cell(row=r, column=time_col).value or "").strip()
            start_t, end_t = parse_time_block(time_cell_val)
            
            # If there's no valid time layout on this row, skip it (structural noise)
            if not start_t:
                continue

            # Loop across the registered students columns for this row
            for col_idx, student_name in students.items():
                cell = sheet.cell(row=r, column=col_idx)
                availability, is_approved = extract_cell_value_and_color(cell)
                
                # # Filter out blank rows entirely to minimize database bloat
                # if availability is None:
                #     print(file_path, current_date, student_name, "skipped", cell, cell.value)
                #     break
                #     continue
                    
                master_records.append({
                    "source_file": file_path,
                    "event_date": current_date,
                    "start_time": start_t,
                    "end_time": end_t,
                    "student_name": student_name,
                    "availability": availability,
                    "is_approved": is_approved
                })
                
    # Build a clean database ready DataFrame
    master_df = pd.DataFrame(master_records)
    return master_df



In [24]:
# --- Execution Entry Point ---
if __name__ == "__main__":
    # Point this path to the directory containing your 19 Excel files
    INPUT_FOLDER = "./data/Schedule" 
    
    final_data = process_all_workbooks(INPUT_FOLDER)

[2026-05-21 13:28:08] starting recursive scan in: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule
Scan complete. Discovered 18 potential Excel files across all subfolders.

Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.08_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R8.3_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.10_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.12_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.09_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.07_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.11_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.06_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_person

In [7]:
# # Export to unified CSV file
final_data.to_csv("master_student_schedule.csv", index=False)
print(f"\nSuccessfully compiled {len(final_data)} neat data rows into master_student_schedule.csv!")


Successfully compiled 12655 neat data rows into master_student_schedule.csv!


In [25]:
type(final_data)

pandas.DataFrame

working report data extraction

In [1]:
import os
import re
import glob
from pathlib import Path
from datetime import datetime, time
import pandas as pd
import openpyxl

In [2]:
def find_books(folder_path):
    root_dir = Path(folder_path)
    
    if not root_dir.exists():
        print(f"[ERROR] The directory path '{folder_path}' does not exist.")
        return []
        
    excel_extensions = {'.xlsx', '.xlsm'}
    discovered_paths = []
    
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] starting recursive scan in: {root_dir.resolve()}")
    
    # rglob("*") recursively yields all files and directories under the root path
    for path in root_dir.rglob("*"):
        # Ensure it's a file, matches our target extensions, and isn't an Excel temporary/lock file
        if path.is_file() and path.suffix.lower() in excel_extensions:
            if not path.name.startswith("~$"):
                # Append the absolute path as a string to match your existing functions
                discovered_paths.append(str(path.resolve()))
                
    print(f"Scan complete. Discovered {len(discovered_paths)} potential Excel files across all subfolders.\n")
    return discovered_paths

In [3]:
find_books("./data/reports")

[2026-05-21 21:16:43] starting recursive scan in: /home/ubuntu/Desktop/GCL/gcl_personnel/data/reports
Scan complete. Discovered 32 potential Excel files across all subfolders.



['/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/SA勤怠管理まとめ表.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Rina(武田 理菜)/2025/Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Rina(武田 理菜)/2026/Rina Takeda業務開始・終了時間記録簿（R8.4～）_式入り.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Taketo (Taketo Shinjo)/2026/Taketo Shinjo業務開始・終了時間記録簿（R8.4～）_式入り.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Pan(PAN WEIZHENG)/2025/PAN WEIZHENG業務開始・終了時間記録簿（R7.4～）_.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Pan(PAN WEIZHENG)/2026/Pan Weizheng業務開始・終了時間記録簿（R8.4～）_式入り.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Anna (Anna Biedermann)/2025/Anna Biedermann業務開始・終了時間記録簿（R7.12～）_.xlsx',
 '/home/ubuntu/Desktop/GCL/gcl_personnel/data/reports/Working Report/Anna (Anna Biedermann)/2026/Anna Biederma

In [36]:
# ------------------------------------------------------------------
# 1. CONFIGURATION & MAPPING
# ------------------------------------------------------------------
FOLDER_PATH = "./data/reports"  # Change this to your directory path
MASTER_BOOK_PATH = "master_student_schedule.csv"

# Name mapping dict based on your table
NAME_MAP = {
    "Aimi Alina Binti Hussin": "Aimi",
    "Anna Biedermann": "Anna",
    "Aoi Koga": "Aoi",
    "Chihiro Iwata": "Chihiro",
    "Hana Miyata": "Hana",
    "Mwaura Jedidah Mweru": "Jedidah",
    "Karimuddin Hakim Hasibuan": "Karim",
    "Kung Gomez Kelvin": "Kelvin",
    "Liu Minglun": "Liu M",
    "Liu Yizhen": "Liu Y",
    "Nao Okumura": "Nao",
    "Natsumi Takushima": "Natsumi",
    "PAN WEIZHENG": "Pan",
    "Roesman Ridwan Raja": "Raja",
    "Rina Takeda": "Rina",
    "Sirpy Palaniswamy": "Sirpy",
    "Hana Sugawara": "Sugawara",
    "Taketo Shinjo": "Taketo"
}

def get_master_name(filename):
    """Extracts the student staff name from the file system name."""
    # Remove common suffixes and clean up spacing
    clean_name = filename.replace("業務開始・終了時間記録簿", "")
    clean_name = re.sub(r'（.*）', '', clean_name) # Remove Japanese era parts like （R7.4～）
    clean_name = clean_name.replace("_式入り", "").replace("_", "").replace(".xlsx", "")
    clean_name = clean_name.strip()
    
    # Match against our strict mapping (case-insensitive lookup helper)
    for full_name, short_name in NAME_MAP.items():
        if full_name.lower() in clean_name.lower():
            return short_name
    return None

def parse_cell_values(cell_value):
    """
    Handles mixed cell types. Returns a list of strings representing values.
    If it's an openpyxl datetime.time object, converts it to 'HH:MM'.
    If it's a multiline string, splits it by linebreaks.
    """
    if cell_value is None:
        return []
    
    if isinstance(cell_value, datetime):
        return [cell_value.strftime("%Y-%m-%d")]
    
    if isinstance(cell_value, (time, datetime)):
        return [cell_value.strftime("%H:%M")]
    
    # Handle numeric values (like hours calculation or day numeric values)
    if isinstance(cell_value, (int, float)):
        return [str(cell_value)]
        
    # Standard text/string handling
    val_str = str(cell_value).strip()
    if not val_str:
        return []
        
    # Split by newline if multiple items exist in one cell
    if "\n" in val_str:
        return [line.strip() for line in val_str.split("\n") if line.strip()]
    return [val_str]

In [41]:
# ------------------------------------------------------------------
# 2. CORE PROCESSING LOOP
# ------------------------------------------------------------------
master_records = []

# Find all Excel files in the target directory
excel_files = find_books(FOLDER_PATH)

for file_path in excel_files:
    filename = os.path.basename(file_path)
    student_name = get_master_name(filename)
    
    if not student_name:
        print(f"⚠️ Skipping file: Could not map student name from '{filename}'")
        continue

    print(f"⏳ Processing: {student_name} ({filename})")
    
    # Load workbook using openpyxl (data_only=True evaluates Excel formula results)
    try:
        wb = openpyxl.load_workbook(file_path, data_only=True)
    except Exception as e:
        print(f"❌ Failed to read {filename}: {e}")
        continue

    # Process month sheets (1月 to 12月)
    for sheet_name in wb.sheetnames:
        if sheet_name == "記入例":
            continue
        
        ws = wb[sheet_name]
        
        # Extract the Month/Year Anchor from cell A5
        date_match = str(ws["A5"].value or "")
            
        # # Data grid loop: Headers occupy 3 rows, so raw rows start at row 4
        # # We search downwards until we hit the '合計' footer
        current_row = 10
        while True:
            cell_a = ws.cell(row=current_row, column=1).value # Col A: Day
            
            # Boundary Check: stop processing if footer encountered or sheet ends completely
            if cell_a is None and ws.cell(row=current_row+1, column=1).value is None:
                break
            if cell_a and any(keyword in str(cell_a) for keyword in ["合計", "Total", "総合計"]):
                break
                
            # Extract raw row values using top-left rules of your geometric column merges
            day_list    = parse_cell_values(ws.cell(row=current_row, column=1).value) # Col A/B
            start_list  = parse_cell_values(ws.cell(row=current_row, column=8).value) # Col H/I/J (Actual Start)
            finish_list = parse_cell_values(ws.cell(row=current_row, column=11).value) # Col K/L/M (Actual Finish)
            report_list = parse_cell_values(ws.cell(row=current_row, column=23).value) # Col W (Report)
            
            # Skip row if no work data is present
            if not start_list or not day_list[0]:
                current_row += 1
                continue
            # Zip or align multiple shifts recorded inside the single text blocks
            max_shifts = max(len(start_list), len(finish_list))
            for i in range(max_shifts):
                try:
                    act_start = start_list[i] if i < len(start_list) else ""
                    act_finish = finish_list[i] if i < len(finish_list) else ""
                    # Combine report rows into one string or map them sequentially if split
                    report_text = report_list[i] if i < len(report_list) else (report_list[0] if report_list else "")
                    
                    # Construct valid timestamp structure for the master book
                    # record_date = f"{year}-{month:02d}-{day_num:02d}"
                    
                    # Append flat tidy row data matching master target headers
                    master_records.append({
                        "source_file": filename,
                        "student_name": student_name,
                        "date": day_list[0],
                        "block_start_time": act_start,
                        "block_end_time": act_finish,
                        "report":report_text
                    })
                except Exception as row_err:
                    print(f"⚠️ Parsing error on row {current_row} inside {filename}: {row_err}")
            
            current_row += 1

[2026-05-21 21:58:59] starting recursive scan in: /home/ubuntu/Desktop/GCL/gcl_personnel/data/reports
Scan complete. Discovered 32 potential Excel files across all subfolders.

⚠️ Skipping file: Could not map student name from 'SA勤怠管理まとめ表.xlsx'
⏳ Processing: Rina (Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx)
⏳ Processing: Rina (Rina Takeda業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Taketo (Taketo Shinjo業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Pan (PAN WEIZHENG業務開始・終了時間記録簿（R7.4～）_.xlsx)
⏳ Processing: Pan (Pan Weizheng業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Anna (Anna Biedermann業務開始・終了時間記録簿（R7.12～）_.xlsx)
⏳ Processing: Anna (Anna Biedermann業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Nao (Nao Okumura 業務開始・終了時間記録簿（R7.4～）_.xlsx)
⏳ Processing: Nao (Nao Okumura業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Liu M (Liu Minglun業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processing: Sugawara (Hana Sugawara 業務開始・終了時間記録簿（R7.4～）_.xlsx)
⏳ Processing: Sugawara (Hana Sugawara業務開始・終了時間記録簿（R8.4～）_式入り.xlsx)
⏳ Processin

In [44]:
master_records

[{'source_file': 'Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
  'student_name': 'Rina',
  'date': '2025-06-10',
  'block_start_time': '12:15',
  'block_end_time': '13:00',
  'report': '5/15スタッフミーティング分'},
 {'source_file': 'Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
  'student_name': 'Rina',
  'date': '2025-06-11',
  'block_start_time': '12:15',
  'block_end_time': '13:00',
  'report': 'スタッフミーテイング'},
 {'source_file': 'Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
  'student_name': 'Rina',
  'date': '2025-06-12',
  'block_start_time': '11:30',
  'block_end_time': '12:45',
  'report': 'イベント準備'},
 {'source_file': 'Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
  'student_name': 'Rina',
  'date': '2025-06-13',
  'block_start_time': '12:00',
  'block_end_time': '14:15',
  'report': 'イベント当日対応'},
 {'source_file': 'Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx',
  'student_name': 'Rina',
  'date': '2025-06-26',
  'block_start_time': '12:30',
  'block_end_time': '14:15',
  'report': '英語課題対応、英会話対応'},
 {'source_file': 'Rina T

In [51]:
# ------------------------------------------------------------------
# 3. EXPORT TO TIDY DATA TARGET
# ------------------------------------------------------------------
if master_records:
    df_master = pd.DataFrame(master_records)
    
    # Reorder columns to align 1:1 with your master book architecture
    headers = ["source_file", "student_name", "date", "block_start_time", "block_end_time", "report"]
    df_master = df_master[headers]
    
    # Save target file output
    df_master.to_excel("student staff reports.xlsx", index=False)
    # print(f"🏁 Success! Created '{MASTER_BOOK_PATH}' with {len(df_master)} transactional data rows.")
else:
    print("❌ Process complete: Zero records were extracted. Please check file grid coordinates.")

In [47]:
df_master

,source_file,student_name,date,block_start_time,block_end_time,report
0,Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx,Rina,2025-06-10,12:15,13:00,5/15スタッフミーティング分
1,Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx,Rina,2025-06-11,12:15,13:00,スタッフミーテイング
2,Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx,Rina,2025-06-12,11:30,12:45,イベント準備
3,Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx,Rina,2025-06-13,12:00,14:15,イベント当日対応
4,Rina Takeda 業務開始・終了時間記録簿（R7.4～）_.xlsx,Rina,2025-06-26,12:30,14:15,英語課題対応、英会話対応
...,...,...,...,...,...,...
812,Kung Gomez Kelvin業務開始・終了時間記録簿（R8.4～）_式入り.xlsx,Kelvin,2026-04-21,12:30,14:00,Leanring about GCL internal works
813,Kung Gomez Kelvin業務開始・終了時間記録簿（R8.4～）_式入り.xlsx,Kelvin,2026-04-22,12:30,13:30,English conversation with Yuki-san
814,Kung Gomez Kelvin業務開始・終了時間記録簿（R8.4～）_式入り.xlsx,Kelvin,2026-04-23,12:30,13:30,English conversation with Yuki-san
815,Kung Gomez Kelvin業務開始・終了時間記録簿（R8.4～）_式入り.xlsx,Kelvin,2026-04-27,12:30,14:00,Filling information on worksheet


In [53]:
import pandas as pd

# ------------------------------------------------------------------
# 1. FILE CONFIGURATION
# ------------------------------------------------------------------
MASTER_BOOK_PATH = "student staff reports.xlsx"
MAIN_BOOK_PATH = "master_student_schedule.csv"
OUTPUT_BOOK_PATH = "final_book.csv"

print("⏳ Loading books...")
df_master = pd.read_excel(MASTER_BOOK_PATH)
df_main = pd.read_csv(MAIN_BOOK_PATH)

# Clean and normalize join keys
for df in [df_master, df_main]:
    df['student_name'] = df['student_name'].astype(str).str.strip()
    df['date'] = df['date'].astype(str).str.strip()

# ------------------------------------------------------------------
# 2. MATCH & INTEGRATE
# ------------------------------------------------------------------
print("🔄 Merging records...")

# Left join using explicit suffixes to keep track of columns cleanly
df_merged = pd.merge(
    df_main, 
    df_master, 
    on=["date", "student_name"], 
    how="left",
    suffixes=("_main", "_master")
)

# ------------------------------------------------------------------
# 3. SCHEMA ALIGNMENT & EXPORT
# ------------------------------------------------------------------
# Map and extract into your explicit target layout
df_final = df_merged[[
    "source_file_main",       # From main_book
    "source_file_master",       # From main_book
    "date",                   # Join Key
    "block_start_time_main",  # From main_book
    "block_end_time_main",    # From main_book
    "student_name",           # Join Key
    "availability",           # From main_book
    "is_approved",            # From main_book
    "block_start_time_master", # From master_book -> Maps to actual_start_time
    "block_end_time_master",   # From master_book -> Maps to actual_end_time
    "report"                  # From master_book
]].copy()

# Rename columns to your exact requested clean schema layout
df_final.columns = [
    "source_file_schedule",
    "source_file_report",
    "date",
    "block_start_time",
    "block_end_time",
    "student_name",
    "availability",
    "is_approved",
    "actual_start_time",
    "actual_end_time",
    "report"
]

# Save tracking result (index=False ensures no row numbers are injected)
df_final.to_csv(OUTPUT_BOOK_PATH, index=False, encoding='utf-8-sig')
print(f"🏁 Integration complete! File written safely to '{OUTPUT_BOOK_PATH}' with {len(df_final)} rows.")

⏳ Loading books...


🔄 Merging records...
🏁 Integration complete! File written safely to 'final_book.csv' with 13213 rows.
